In [7]:
from Sequential_Fish.chromatic_abberrations import load_calibration, apply_polynomial_transform_spots,apply_polynomial_transform_to_signal , get_polynomial_features, CALIBRATION_FOLDER
from Sequential_Fish.chromatic_abberrations.calibration import _load_calibration_index
from Sequential_Fish.tools.utils import open_image
calibration_image_path = "/media/SSD_floricslimani/Fish_seq/Davide/Chromatic abberations/Beads-Field_01.tif"

In [8]:
import pandas as pd
Spots = pd.read_feather(test_path)
Spots

IsADirectoryError: [Errno 21] est un dossier: '/media/SSD_floricslimani/Fish_seq/Davide/2024-08-12 - SeqFISH - HeLa - Puro - R2TP1-2_Run7/result_tables/'

In [11]:
calibration = load_calibration(555,640)
calibration

{'x_fit': LinearRegression(),
 'y_fit': LinearRegression(),
 'z_fit': LinearRegression(),
 'polynomial_features': PolynomialFeatures(),
 'polynomial_features_inv': PolynomialFeatures(),
 'x_inv_fit': LinearRegression(),
 'y_inv_fit': LinearRegression(),
 'z_inv_fit': LinearRegression(),
 'voxel_size': (200, 97, 97),
 'degree': 2,
 'reference_wavelength': 555,
 'corrected_wavelength': 640,
 'timestamp': '10_07_2025_14_08_23'}

# Check on calibration image : 
--> Check that model are still correctly fitted afte saving with joblib

In [12]:
calibration_image = open_image(calibration_image_path)
calibration_image.shape

(47, 1981, 2004, 3)

In [ ]:
corrected_image = apply_polynomial_transform_to_signal(
    image= calibration_image[...,2],
    poly= calibration['polynomial_features_inv'],
    model_x= calibration['x_inv_fit'],
    model_y= calibration['y_inv_fit'],
    model_z= calibration['z_inv_fit'],
    voxel_size=calibration['voxel_size']
)

corrected_image_2d = apply_polynomial_transform_to_signal(
    image= calibration_image[...,2],
    poly= calibration['polynomial_features_inv'],
    model_x= calibration['x_inv_fit'],
    model_y= calibration['y_inv_fit'],
    voxel_size=calibration['voxel_size']
)

(186586428, 3)


In [ ]:
import napari, os
os.environ["QT_QPA_PLATFORM"] = "xcb"

voxel_size = calibration['voxel_size']

Viewer = napari.Viewer()
Viewer.add_image(calibration_image[...,1], scale= voxel_size, name= "reference", colormap= "green", blending= "additive")
Viewer.add_image(calibration_image[...,2], scale= voxel_size, name= "abeeration", colormap= "red", blending= "additive")
Viewer.add_image(corrected_image, scale= voxel_size, name= "corrected_image", colormap= "blue", blending= "additive")
Viewer.add_image(corrected_image_2d, scale= voxel_size, name= "corrected_image_2d", colormap= "blue", blending= "additive")

napari.run()

# Check on spots

In [ ]:
def convert_str_coordinates_to_zyx_columns(coordinates_series : pd.Series) :
    coordinates_series = coordinates_series.str.replace('(','').str.replace(')','').str.split(',')
    res_df = pd.DataFrame(coordinates_series)
    res_df['z'] = res_df['coordinates'].apply(lambda x : int(x[0])).astype(int)
    res_df['y'] = res_df['coordinates'].apply(lambda x : int(x[2])).astype(int)
    res_df['x'] = res_df['coordinates'].apply(lambda x : int(x[1])).astype(int)

    return res_df


In [ ]:
import pandas as pd
import numpy as np
spots_path = "/media/SSD_floricslimani/Fish_seq/Davide/Chromatic abberations/"
ref_spots = pd.read_csv(spots_path + "/Beads-Field_01_channel1.csv", sep= ";")['coordinates'] #555nm
abb_spots = pd.read_csv(spots_path + "/Beads-Field_01_channel2.csv", sep= ";")['coordinates'] #640nm
ref_spots = convert_str_coordinates_to_zyx_columns(ref_spots)
abb_spots = convert_str_coordinates_to_zyx_columns(abb_spots)
ref_spots

,coordinates,z,y,x
0,"[17, 106, 1888]",17,1888,106
1,"[18, 65, 1819]",18,1819,65
2,"[18, 77, 1738]",18,1738,77
3,"[18, 1944, 23]",18,23,1944
4,"[19, 59, 1768]",19,1768,59
...,...,...,...,...
527,"[32, 1079, 1031]",32,1031,1079
528,"[45, 1339, 1775]",45,1775,1339
529,"[46, 339, 1249]",46,1249,339
530,"[46, 781, 1796]",46,1796,781


In [ ]:
abb_coordinates = np.array(list(
    zip(abb_spots['z'],abb_spots['y'],abb_spots['x'],)
)).astype(int)

ref_coordinates = np.array(list(
    zip(ref_spots['z'],ref_spots['y'],ref_spots['x'],)
)).astype(int)

ref_coordinates.shape

(532, 3)

In [ ]:
spots_corrected = apply_polynomial_transform_spots(
    abb_coordinates,
    poly=calibration['polynomial_features_inv'],
    model_x=calibration['x_inv_fit'],
    model_y=calibration['y_inv_fit'],
    model_z=calibration['z_inv_fit'],
    voxel_size=calibration['voxel_size']
)

spots_2D_corrected = apply_polynomial_transform_spots(
    abb_coordinates,
    poly=calibration['polynomial_features'],
    model_x=calibration['x_inv_fit'],
    model_y=calibration['y_inv_fit'],
    voxel_size=calibration['voxel_size']
)

In [ ]:
import napari, os
os.environ["QT_QPA_PLATFORM"] = "xcb"

voxel_size = calibration['voxel_size']

Viewer = napari.Viewer()
Viewer.add_image(calibration_image[...,1], scale= voxel_size, name= "reference", colormap= "green", blending= "additive")
Viewer.add_image(calibration_image[...,2], scale= voxel_size, name= "abeeration", colormap= "red", blending= "additive")
Viewer.add_image(corrected_image, scale= voxel_size, name= "corrected_image", colormap= "blue", blending= "additive")
Viewer.add_image(corrected_image_2d, scale= voxel_size, name= "corrected_image_2d", colormap= "blue", blending= "additive")

Viewer.add_points(ref_coordinates, scale= voxel_size, name= "ref_spots", blending="additive", face_color= "transparent")
Viewer.add_points(spots_corrected, scale= voxel_size, name= "spots_corrected", blending="additive", face_color= "transparent")
Viewer.add_points(spots_2D_corrected, scale= voxel_size, name= "spots_2D_corrected", blending="additive", face_color= "transparent")
Viewer.add_points(abb_coordinates, scale= voxel_size, name= "abb_coordinates", blending="additive", face_color= "transparent")

napari.run()

# Check on DataFrame

In [ ]:
import pandas as pd
test_path = "/media/SSD_floricslimani/Fish_seq/Davide/2024-08-12 - SeqFISH - HeLa - Puro - R2TP1-2_Run7/result_tables/"
Spots = pd.read_feather(test_path + "Spots.feather")
Detection = pd.read_feather(test_path + "Detection.feather")

In [ ]:
Spots

,spot_id,cluster_id,drifted_z,drifted_y,drifted_x,intensity,population,detection_id,acquisition_id,drift_z,...,x_shape,z,y,x,cycle,color_id,is_washout,coordinates,in_nucleus,cell_label
0,0,nan,5,93,1600,8120,free,1,0,0,...,2004,5,93,1600,0,0,False,"[5, 93, 1600]",True,1.0
1,1,nan,5,123,1625,8751,free,1,0,0,...,2004,5,123,1625,0,0,False,"[5, 123, 1625]",True,1.0
2,2,nan,5,128,1634,8427,free,1,0,0,...,2004,5,128,1634,0,0,False,"[5, 128, 1634]",True,1.0
3,3,nan,5,141,610,8403,free,1,0,0,...,2004,5,141,610,0,0,False,"[5, 141, 610]",True,4.0
4,4,nan,5,157,1650,8633,free,1,0,0,...,2004,5,157,1650,0,0,False,"[5, 157, 1650]",True,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374065,229359,nan,37,301,290,4426,free,140,121,0,...,2004,37,301,290,13,1,True,"[37, 301, 290]",False,0.0
374066,229360,nan,14,1902,1854,4059,free,140,121,0,...,2004,14,1902,1854,13,1,True,"[14, 1902, 1854]",False,0.0
374067,264241,nan,15,1549,595,7244,free,168,122,0,...,2004,15,1549,595,13,1,True,"[15, 1549, 595]",False,32.0
374068,312150,nan,17,505,864,6322,free,196,123,0,...,2004,17,505,864,13,1,True,"[17, 505, 864]",True,11.0


In [ ]:
Detection

,detection_id,acquisition_id,visual_name,filename,voxel_size,spot_size,alpha,beta,gamma,artifact_radius,cluster_size,min_spot_per_cluster,Threshold_0,Threshold_1,threshold,color_id,image_path,image_key,location,wavelength
0,1,0,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,"[200, 97, 97]","[300, 140, 140]",0.5,1,2,1400,300,3,400,400,400,0,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,1,Location-01,555
1,15,0,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,"[200, 97, 97]","[300, 140, 140]",0.5,1,2,1400,300,3,400,400,400,1,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,15,Location-01,640
2,2,9,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,"[200, 97, 97]","[300, 140, 140]",0.5,1,2,1400,300,3,598,348,598,0,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,2,Location-01,555
3,16,9,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,"[200, 97, 97]","[300, 140, 140]",0.5,1,2,1400,300,3,598,348,348,1,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,16,Location-01,640
4,3,18,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,"[200, 97, 97]","[300, 140, 140]",0.5,1,2,1400,300,3,598,348,598,0,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,3,Location-01,555
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247,250,107,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,"[200, 97, 97]","[300, 140, 140]",0.5,1,2,1400,300,3,500,240,240,1,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,250,Location-10,640
248,237,116,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,"[200, 97, 97]","[300, 140, 140]",0.5,1,2,1400,300,3,500,330,500,0,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,237,Location-10,555
249,251,116,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,"[200, 97, 97]","[300, 140, 140]",0.5,1,2,1400,300,3,500,330,330,1,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,251,Location-10,640
250,238,125,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,"[200, 97, 97]","[300, 140, 140]",0.5,1,2,1400,300,3,500,330,500,0,/mnt/ssd/SSD_floricslimani/Fish_seq/Davide/202...,238,Location-10,555


In [ ]:
from Sequential_Fish.chromatic_abberrations import correct_Spots_dataframe

correct_Spots_dataframe(
    Detection=Detection,
    Spots=Spots,
    reference_wavelength=555
)

0            [5, 93, 1600]
1           [5, 123, 1625]
2           [5, 128, 1634]
3            [5, 141, 610]
4           [5, 157, 1650]
                ...       
374065      [37, 301, 290]
374066    [14, 1902, 1854]
374067     [15, 1549, 595]
374068      [17, 505, 864]
374069    [34, 1946, 1798]
Name: coordinates, Length: 374070, dtype: object
0            [5, 93, 1600]
1           [5, 123, 1625]
2           [5, 128, 1634]
3            [5, 141, 610]
4           [5, 157, 1650]
                ...       
374065      [36, 301, 292]
374066    [18, 1900, 1852]
374067     [20, 1548, 595]
374068      [22, 506, 864]
374069    [32, 1941, 1796]
Name: coordinates, Length: 374070, dtype: object


,spot_id,cluster_id,drifted_z,drifted_y,drifted_x,intensity,population,detection_id,acquisition_id,drift_z,...,x_shape,z,y,x,cycle,color_id,is_washout,coordinates,in_nucleus,cell_label
0,0,nan,5,93,1600,8120,free,1,0,0,...,2004,5,93,1600,0,0,False,"[5, 93, 1600]",True,1.0
1,1,nan,5,123,1625,8751,free,1,0,0,...,2004,5,123,1625,0,0,False,"[5, 123, 1625]",True,1.0
2,2,nan,5,128,1634,8427,free,1,0,0,...,2004,5,128,1634,0,0,False,"[5, 128, 1634]",True,1.0
3,3,nan,5,141,610,8403,free,1,0,0,...,2004,5,141,610,0,0,False,"[5, 141, 610]",True,4.0
4,4,nan,5,157,1650,8633,free,1,0,0,...,2004,5,157,1650,0,0,False,"[5, 157, 1650]",True,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374065,229359,nan,37,301,290,4426,free,140,121,0,...,2004,36,301,292,13,1,True,"[36, 301, 292]",False,0.0
374066,229360,nan,14,1902,1854,4059,free,140,121,0,...,2004,18,1900,1852,13,1,True,"[18, 1900, 1852]",False,0.0
374067,264241,nan,15,1549,595,7244,free,168,122,0,...,2004,20,1548,595,13,1,True,"[20, 1548, 595]",False,32.0
374068,312150,nan,17,505,864,6322,free,196,123,0,...,2004,22,506,864,13,1,True,"[22, 506, 864]",True,11.0


# Test on run 7

In [2]:
import pandas as pd
import numpy as np

In [ ]:
path = ""
Detection = pd.read_feather(path + "/Detection.feather")
Spots = pd.read_feather(path + "/Spots.feather")